# Text Cleaning

Extracts and cleans text from the downloaded documents, and builds the master text dataset used by all downstream stages.

PDF is used as the primary source. Inspection of both formats (this notebook) showed RBI's HTML pages embed the site's full navigation menu (~280 lines) before the substantive content begins, while the PDFs carry only a short bilingual masthead — substantially less boilerplate to remove. The cleaning pipeline (`src/utils.py`) strips header boilerplate at an anchor point (RBI's contact email, which is constant across the sample, with a plain-date fallback for the small number of documents where it does not appear) and footer boilerplate at RBI's internal filing-number pattern (`Press Release: YYYY-YYYY/NNN`).

## Step 0 — Path setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\hp\Desktop\rbi-sentiment-market-forecast


## Step 1 — Sample selection

Selects one recent Resolution, Governor's Statement, and Minutes document for inspection.

In [2]:
import pandas as pd
from paths import DOWNLOAD_MANIFEST_CSV

manifest = pd.read_csv(DOWNLOAD_MANIFEST_CSV)

samples = manifest[manifest["doc_type"].isin(["resolution", "minutes", "governor_statement"])]
samples = samples.sort_values("listed_date").groupby("doc_type").tail(1)
samples[["fin_year", "listed_date", "title", "doc_type", "html_path", "pdf_path"]]

,fin_year,listed_date,title,doc_type,html_path,pdf_path
130,2024-2025,"Oct 23, 2024",Minutes of the Monetary Policy Committee Meeti...,minutes,C:\Users\hp\Desktop\rbi-sentiment-market-forec...,C:\Users\hp\Desktop\rbi-sentiment-market-forec...
92,2022-2023,"Sep 30, 2022",Governor’s Statement,governor_statement,C:\Users\hp\Desktop\rbi-sentiment-market-forec...,C:\Users\hp\Desktop\rbi-sentiment-market-forec...
93,2022-2023,"Sep 30, 2022",Resolution of the Monetary Policy Committee (M...,resolution,C:\Users\hp\Desktop\rbi-sentiment-market-forec...,C:\Users\hp\Desktop\rbi-sentiment-market-forec...


## Step 2 — Raw HTML extraction (Resolution)

In [3]:
from bs4 import BeautifulSoup

res_row = samples[samples["doc_type"] == "resolution"].iloc[0]
print("File:", res_row["title"], "|", res_row["listed_date"])
print("=" * 70)

html = open(res_row["html_path"], encoding="utf-8").read()
soup = BeautifulSoup(html, "lxml")

raw_text = soup.get_text(separator="\n", strip=True)
print(f"Total length: {len(raw_text)} characters\n")
print(raw_text[:3000])

File: Resolution of the Monetary Policy Committee (MPC) September 28-30, 2022 | Sep 30, 2022
Total length: 15032 characters

Press Releases | Official Website of Reserve Bank of India
Skip to main content
Not Pressed
Not Pressed
Pressed
Not Pressed
Not Pressed
Pressed
Not Pressed
Change Language
हिंदी
Search the Website
Search
Home
About Us ▼
About Us
Organisation & Functions
▶
Organisation Structure
Departments
Offices
Training Establishment
▶
College of Agricultural Banking
Reserve Bank Staff College
College of Supervisors
RBI's Functions and Working
Governors
Deputy Governors
Executive Directors
Media Kit
Communication Policy of RBI
Sources of Information
▶
Annual Publications
Half-yearly Publications
Quarterly Publications
Monthly Publications
Weekly Publications
Occasional Publications
SDDS
NSDP
Data Releases
Publications available on Subscription
General Information
RBI History
Museum
▶
The RBI Museum
RBI Monetary Museum
Notification ▼
Notifications
Master Directions
Master Circu

## Step 3 — Raw PDF extraction (same document)

Compares extraction quality between the two source formats.

In [4]:
import pdfplumber

with pdfplumber.open(res_row["pdf_path"]) as pdf:
    pdf_text = "\n".join(page.extract_text() or "" for page in pdf.pages)

print(f"Total length: {len(pdf_text)} characters | {len(pdf.pages)} pages\n")
print(pdf_text[:3000])

Total length: 9095 characters | 4 pages

�ेस �काशनी PRESS RELEASE
भारतीय �रज़व� ब�क
RESERVE BANK OF INDIA
वेबसाइट : www.rbi.org.in/hindi सचं ार िवभाग, क��ीय कायार्लय, एस.बी.एस. माग,र् फोटर्, मबुं ई - 400 001
0
Website : www.rbi.org.in Department of Communication, Central Office, S.B.S. Marg, Fort, Mumbai - 400 001
ई-मले /email : helpdoc@rbi.org.in फोन/Phone: 022 - 2266 0502
September 30, 2022
Monetary Policy Statement, 2022-23
Resolution of the Monetary Policy Committee (MPC)
September 28-30, 2022
On the basis of an assessment of the current and evolving macroeconomic
situation, the Monetary Policy Committee (MPC) at its meeting today (September 30,
2022) decided to:
• Increase the policy repo rate under the liquidity adjustment facility (LAF) by 50
basis points to 5.90 per cent with immediate effect.
Consequently, the standing deposit facility (SDF) rate stands adjusted to 5.65 per
cent and the marginal standing facility (MSF) rate and the Bank Rate to 6.15 per
cent.
• The MPC also dec

## Step 4 — Raw HTML extraction (Governor's Statement, Minutes)

In [5]:
gov_row = samples[samples["doc_type"] == "governor_statement"].iloc[0]
print("File:", gov_row["title"], "|", gov_row["listed_date"])
print("=" * 70)

html = open(gov_row["html_path"], encoding="utf-8").read()
soup = BeautifulSoup(html, "lxml")
raw_text = soup.get_text(separator="\n", strip=True)
print(f"Total length: {len(raw_text)} characters\n")
print(raw_text[:3000])

File: Governor’s Statement | Sep 30, 2022
Total length: 28053 characters

Press Releases | Official Website of Reserve Bank of India
Skip to main content
Not Pressed
Not Pressed
Pressed
Not Pressed
Not Pressed
Pressed
Not Pressed
Change Language
हिंदी
Search the Website
Search
Home
About Us ▼
About Us
Organisation & Functions
▶
Organisation Structure
Departments
Offices
Training Establishment
▶
College of Agricultural Banking
Reserve Bank Staff College
College of Supervisors
RBI's Functions and Working
Governors
Deputy Governors
Executive Directors
Media Kit
Communication Policy of RBI
Sources of Information
▶
Annual Publications
Half-yearly Publications
Quarterly Publications
Monthly Publications
Weekly Publications
Occasional Publications
SDDS
NSDP
Data Releases
Publications available on Subscription
General Information
RBI History
Museum
▶
The RBI Museum
RBI Monetary Museum
Notification ▼
Notifications
Master Directions
Master Circulars
Amendment Directions
Draft Notifications/Guide

In [6]:
min_row = samples[samples["doc_type"] == "minutes"].iloc[0]
print("File:", min_row["title"], "|", min_row["listed_date"])
print("=" * 70)

html = open(min_row["html_path"], encoding="utf-8").read()
soup = BeautifulSoup(html, "lxml")
raw_text = soup.get_text(separator="\n", strip=True)
print(f"Total length: {len(raw_text)} characters\n")
print(raw_text[:3000])

File: Minutes of the Monetary Policy Committee Meeting, October 7 to 9, 2024 | Oct 23, 2024
Total length: 47815 characters

Press Releases | Official Website of Reserve Bank of India
Skip to main content
Not Pressed
Not Pressed
Pressed
Not Pressed
Not Pressed
Pressed
Not Pressed
Change Language
हिंदी
Search the Website
Search
Home
About Us ▼
About Us
Organisation & Functions
▶
Organisation Structure
Departments
Offices
Training Establishment
▶
College of Agricultural Banking
Reserve Bank Staff College
College of Supervisors
RBI's Functions and Working
Governors
Deputy Governors
Executive Directors
Media Kit
Communication Policy of RBI
Sources of Information
▶
Annual Publications
Half-yearly Publications
Quarterly Publications
Monthly Publications
Weekly Publications
Occasional Publications
SDDS
NSDP
Data Releases
Publications available on Subscription
General Information
RBI History
Museum
▶
The RBI Museum
RBI Monetary Museum
Notification ▼
Notifications
Master Directions
Master Circul

---

**Findings.** PDF extraction is substantially cleaner than HTML for this corpus. The remainder of the cleaning pipeline (header/footer boilerplate removal, page-number-artifact stripping) is implemented in `src/utils.py` and applied across the full corpus in later stages of this notebook (not shown here — see the `master_statements.csv` build step and the `extract_document_text()` function).